# Limpeza de Dados de Vendas

Ficheiro: `dados_sujos_limpeza.csv`

> Nota: `read.csv()` e `is.na()` são funções do R. Em Python (pandas) usamos `pd.read_csv()` e `.isna()`.

## 0. Carregar o ficheiro para o Colab
Execute a célula e escolha o ficheiro `dados_sujos_limpeza.csv` no seu computador.

In [ ]:
import re
import pandas as pd

try:
    from google.colab import files
    enviado = files.upload()
    ENTRADA = list(enviado.keys())[0]
except ImportError:
    ENTRADA = "dados_sujos_limpeza.csv"   # fora do Colab
print("Ficheiro:", ENTRADA)

## 1. Importar o ficheiro

In [ ]:
df = pd.read_csv(ENTRADA, encoding="utf-8")
print("Dimensão original:", df.shape)
df.head()

## 2. Identificar valores ausentes
Strings vazias ou só com espaços também contam como ausentes.

In [ ]:
df = df.replace(r"^\s*$", pd.NA, regex=True)
df.isna().sum()

In [ ]:
# Linhas com pelo menos um valor ausente
df[df.isna().any(axis=1)]

## 3. Eliminar registos duplicados

In [ ]:
print("Duplicados encontrados:", df.duplicated().sum())
display(df[df.duplicated(keep=False)].sort_values("id_venda"))

df = df.drop_duplicates()
df = df.drop_duplicates(subset="id_venda", keep="first")  # id deve ser único
print("Dimensão após remover duplicados:", df.shape)

## 4. Corrigir espaços e maiúsculas/minúsculas

In [ ]:
print("ANTES:")
for col in ["cidade", "produto", "metodo_pagamento"]:
    print(f"  {col}: {df[col].dropna().unique().tolist()}")

for col in ["nome_cliente", "cidade", "produto", "email", "metodo_pagamento"]:
    df[col] = df[col].astype("string").str.strip()

df["nome_cliente"] = df["nome_cliente"].str.title()
df["cidade"] = df["cidade"].str.title()          # MATOLA -> Matola
df["produto"] = df["produto"].str.capitalize()   # " computador " -> Computador
df["email"] = df["email"].str.lower()

mapa_pag = {"m-pesa": "M-Pesa", "e-mola": "e-Mola", "cartão": "Cartão", "dinheiro": "Dinheiro"}
df["metodo_pagamento"] = df["metodo_pagamento"].str.lower().map(mapa_pag)

print("\nDEPOIS:")
for col in ["cidade", "produto", "metodo_pagamento"]:
    print(f"  {col}: {sorted(df[col].dropna().unique())}")

## 5. Corrigir tipos de dados
(Feito antes dos outliers: precisamos de números para calcular estatísticas.)

In [ ]:
print("Tipos antes:\n", df.dtypes, "\n")

# Preço com vírgula decimal: "36519,66" -> 36519.66
df["preco_unitario_MZN"] = pd.to_numeric(
    df["preco_unitario_MZN"].astype("string").str.replace(",", ".", regex=False), errors="coerce")
for col in ["idade", "quantidade", "satisfacao"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["id_venda"] = df["id_venda"].astype(int)

# Datas: AAAA-MM-DD, alternativa DD-MM-AAAA; inválidas -> NaT
datas = pd.to_datetime(df["data_venda"], format="%Y-%m-%d", errors="coerce")
datas = datas.fillna(pd.to_datetime(df["data_venda"], format="%d-%m-%Y", errors="coerce"))
print("Datas inválidas:", df.loc[datas.isna() & df["data_venda"].notna(), "data_venda"].tolist())
df["data_venda"] = datas

## 6. Identificar e tratar outliers

In [ ]:
df[["idade", "quantidade", "preco_unitario_MZN", "satisfacao"]].describe()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for a, col in zip(ax, ["idade", "quantidade", "preco_unitario_MZN", "satisfacao"]):
    a.boxplot(df[col].dropna()); a.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
# Regras de negócio
m = ~df["idade"].between(18, 100) & df["idade"].notna()
print("Idade fora de [18,100]:", df.loc[m, "idade"].tolist()); df.loc[m, "idade"] = pd.NA

m = df["quantidade"] <= 0
print("Quantidade <= 0:", df.loc[m, "quantidade"].tolist()); df.loc[m, "quantidade"] = pd.NA

m = ~df["satisfacao"].between(1, 5) & df["satisfacao"].notna()
print("Satisfação fora de [1,5]:", df.loc[m, "satisfacao"].tolist()); df.loc[m, "satisfacao"] = pd.NA

# Preço: método IQR
q1, q3 = df["preco_unitario_MZN"].quantile([0.25, 0.75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
m = ~df["preco_unitario_MZN"].between(lim_inf, lim_sup) & df["preco_unitario_MZN"].notna()
print(f"Preço fora de [{lim_inf:.2f}, {lim_sup:.2f}]:", df.loc[m, "preco_unitario_MZN"].tolist())
df.loc[m, "preco_unitario_MZN"] = pd.NA

### Imputação dos valores em falta (mediana)

In [ ]:
df["preco_unitario_MZN"] = df["preco_unitario_MZN"].fillna(
    df.groupby("produto")["preco_unitario_MZN"].transform("median")).round(2)
for col in ["idade", "quantidade", "satisfacao"]:
    df[col] = df[col].fillna(df[col].median())
df["cidade"] = df["cidade"].fillna("Desconhecida")

for col in ["idade", "quantidade", "satisfacao"]:
    df[col] = df[col].astype(int)
df.dtypes

## 7. Validar emails

In [ ]:
padrao = re.compile(r"^[\w.+-]+@[\w-]+(\.[\w-]+)+$")   # \w aceita acentos
df["email_valido"] = df["email"].apply(lambda e: bool(padrao.match(e)) if pd.notna(e) else False)

print("Emails inválidos:", df.loc[~df["email_valido"] & df["email"].notna(), "email"].tolist())
print("Emails em falta:", df["email"].isna().sum())
df.loc[~df["email_valido"], "email"] = pd.NA

## 8. Criar novo CSV com os dados limpos

In [ ]:
df["valor_total_MZN"] = (df["quantidade"] * df["preco_unitario_MZN"]).round(2)
df = df.sort_values("id_venda").reset_index(drop=True)

SAIDA = "dados_limpos.csv"
df.to_csv(SAIDA, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
print("Dimensão final:", df.shape)
print("\nAusentes restantes:\n", df.isna().sum())
df.head(10)

In [ ]:
# Descarregar o ficheiro limpo (só no Colab)
try:
    from google.colab import files
    files.download(SAIDA)
except ImportError:
    print("Ficheiro gravado em", SAIDA)